# RQ1: Is AI Artifact Adoption Structured or Ad Hoc?

*Do repositories exhibit distinct, repeatable artifact profiles, or is adoption idiosyncratic?*

**Paper thesis**: AI tool adoption in repositories follows a cumulative maturity progression (L1→L4), identifiable from artifact configurations.

**Framework**: AIME (AI Adoption Maturity Evaluator) — 9 semantic categories mapped to 4 maturity levels.

| Sub-question | What it tests |
|---|---|
| **1a.** Category co-occurrence & breadth | Whether adoption is structured as cumulative broadening — do repos build outward from an L2 foundation? |
| **1b.** Cumulative maturity | Whether levels are a *ladder* or a *menu* — do L4 repos contain L2+L3 artifacts? |
| **1c.** Population distribution | Among repos with detectable AI artifacts, what does the maturity landscape look like? |

**Sampling frame**: the full 441-repo private frame (27 orgs) after QA filtering. Repos with strict+W+ artifacts are scored; the 231 frame repos with no strict+W+ artifact are included as **L1** (padding cell in Phase 1) rather than excluded, so population-level proportions are interpretable.

**Data source**: file-level predictions dataset `data/rq1_file_predictions.parquet` (the private frame classified by the multi-signal AIME pipeline from precomputed embeddings — 2,874 file rows across 393 repos; the remaining frame repos have no candidate files — the RQ counterpart of `msrc_file_predictions.parquet` from notebooks 13/14), restricted to **strict+W+ AI artifacts**: the whitelist definition quality-validated in notebooks 13/14 (`discovery_step ∈ {tool_standard, shared_in_tool_folder, shared_in_root}`) **plus W+ exact-basename recoveries** (the notebook-13 Section 5 rule: nested `CLAUDE.md`, `AGENTS.md`, `copilot-instructions.md`, mcp configs, etc.). Discovery yields **1,046 artifacts across 217 repos (23 orgs)**; boilerplate/doc-folder pre-filters leave **1,026 scoreable files across 210 scored repos**. With the 231 padded L1 repos this gives the full 441-repo population; the AI-tools-only subset (N = 217) serves as the robustness comparison.

In [ ]:
# IMPORTANT: Set OpenMP environment variables BEFORE any other imports to prevent kernel crash on macOS ARM
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Data was converted to boolean")
import json
import pickle
import gc
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
import scipy.stats as stats

# Use static image renderer for PDF/HTML export compatibility
# In interactive mode (Jupyter), charts are still interactive;
# nbconvert uses the default renderer which kaleido handles.
pio.renderers.default = "notebook"

# Add project root to path
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.embedding_generator import load_embedding_model, DEFAULT_TASK_PREFIX, DEFAULT_MODEL
from src.maturity_scorer import (
    CATEGORY_NAMES,
    CATEGORY_TO_LEVEL,
    CATEGORY_TEMPLATES,
    MATURITY_LABELS,
    MaturityLevel,
    embed_category_templates,
    classify_by_tool_detection,
    classify_by_path,
    classify_by_content,
    combine_signals,
    aggregate_repo_maturity,
    score_from_output_dir,
    FileClassification,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Categories ({len(CATEGORY_NAMES)}): {CATEGORY_NAMES}")
print(f"Category → Level mapping:")
for cat, lvl in sorted(CATEGORY_TO_LEVEL.items(), key=lambda x: (x[1], x[0])):
    print(f"  {cat} → L{lvl}")

## Configuration

In [ ]:
# Paths
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
ARTIFACTS_DIR = str(PROJECT_ROOT / "Artifacts")
CACHE_PATH = DATA_DIR / "rq1_repo_scores.csv"  # Cache batch results

# Strict AI-artifact filter (same WL_STEPS definition as notebooks 13/14):
# keep only files discovered via the whitelisted AI-tool tiers; the catch-all
# markdown tiers (non_standard_root, non_standard_other) are excluded.
STRICT_MODE = True
WL_STEPS = {"tool_standard", "shared_in_tool_folder", "shared_in_root"}

# W+ recovery (same rule as notebook 13 Section 5): additionally admit
# non-standard files whose EXACT basename is a declared AI artifact —
# nested CLAUDE.md / AGENTS.md / mcp configs that the collector's pattern
# tiers only catch at the root or in tool config folders. Deliberately not
# is_protected_artifact(), whose substring rules admit false positives
# (zabbix-agent docs, agent-framework tutorials).
WL_PLUS = True
from src.artifact_filtering import load_protected_patterns
PROTECTED_EXACT, _, _ = load_protected_patterns(ARTIFACTS_DIR)

def wl_mask(df, name_col="artifact_name"):
    m = df["discovery_step"].isin(WL_STEPS)
    if WL_PLUS:
        m |= df[name_col].str.lower().isin(PROTECTED_EXACT)
    return m

# The scorer's CATEGORY_TEMPLATES now include the "general-documentation"
# absorber category, which has no maturity level (it collapses to
# not-artifact). All level-based analyses below operate on the 9 leveled
# categories only.
CATEGORY_NAMES = [c for c in CATEGORY_NAMES if c in CATEGORY_TO_LEVEL]

# Load filtered metadata to know which repos passed QA
filtered_meta = pd.read_csv(DATA_DIR / "filtered_common_metadata.csv")
if STRICT_MODE:
    filtered_meta = filtered_meta[wl_mask(filtered_meta)].reset_index(drop=True)
print(f"Filtered dataset ({'strict+W+' if (STRICT_MODE and WL_PLUS) else ('strict' if STRICT_MODE else 'full')}): {len(filtered_meta)} artifacts across "
      f"{filtered_meta['repo_name'].nunique()} repos, "
      f"{filtered_meta['org_name'].nunique()} orgs")

# Load AI-tools-only metadata for robustness checks
filtered_ai_meta = pd.read_csv(DATA_DIR / "filtered_ai_tools_metadata.csv")
if STRICT_MODE:
    filtered_ai_meta = filtered_ai_meta[wl_mask(filtered_ai_meta)].reset_index(drop=True)
ai_tool_repos = set(filtered_ai_meta['repo_name'].unique())
print(f"AI-tools subset: {len(filtered_ai_meta)} artifacts across "
      f"{len(ai_tool_repos)} repos (repos with ≥1 known AI tool artifact)")

# Build set of valid (org, repo) pairs from filtered data
# repo_name format: "org_name/repo_name"
filtered_repos = set(filtered_meta['repo_name'].unique())
print(f"Repos to score: {len(filtered_repos)}")

# Level colors for consistent charts
LEVEL_COLORS = {1: "#6b7280", 2: "#3b82f6", 3: "#f97316", 4: "#22c55e"}
LEVEL_LABELS = {1: "L1 Ad Hoc", 2: "L2 Grounded", 3: "L3 Agent-Augmented", 4: "L4 Orchestration"}

# Figures directory for saving charts
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

def save_fig(fig, filename, width=1200, height=600):
    stem = Path(filename).stem
    for ext, kwargs in ((".png", {"scale": 2}), (".pdf", {})):
        path = FIGURES_DIR / f"{stem}{ext}"
        try:
            fig.write_image(str(path), width=width, height=height, **kwargs)
            print(f"Saved: {path}")
        except Exception as e:
            print(f"Could not save {path}: {e}")


## Phase 1: Batch Maturity Scoring

Repo-level AIME scores are derived from the **file-level predictions dataset**
`data/rq1_file_predictions.parquet` (generated by `scripts/regen_rq1_file_predictions.py`,
the RQ counterpart of the MSRC `msrc_file_predictions.parquet` used in notebooks 13/14):
every file in every repo is classified by the multi-signal pipeline, then the strict
whitelist population is selected and aggregated with `aggregate_repo_maturity` —
the exact methodology quality-validated in notebook 13. Results are cached to
`data/rq1_repo_scores.csv`.


In [ ]:
if CACHE_PATH.exists():
    print(f"Loading cached scores from {CACHE_PATH}")
    repo_scores_df = pd.read_csv(CACHE_PATH)
    print(f"Loaded {len(repo_scores_df)} repo scores from cache")
else:
    print("No cache — deriving repo scores from data/rq1_file_predictions.parquet")
    pred = pd.read_parquet(DATA_DIR / "rq1_file_predictions.parquet")
    print(f"File predictions: {len(pred):,} rows across {pred['repo'].nunique()} repos")

    pred["basename"] = pred["artifact_path"].str.split("/").str[-1].str.lower()
    pred["wl_strict"] = pred["discovery_step"].isin(WL_STEPS)
    if WL_PLUS:
        pred["wl_strict"] |= pred["basename"].isin(PROTECTED_EXACT)
    scored_pop = pred[pred["wl_strict"]] if STRICT_MODE else pred
    print(f"Scored population ({'strict+W+' if (STRICT_MODE and WL_PLUS) else ('strict' if STRICT_MODE else 'full')}): "
          f"{len(scored_pop):,} files across {scored_pop['repo'].nunique()} repos")

    def fc_list(sub):
        # Same FileClassification construction as notebook 13 cell 16.
        out = []
        for r in sub.itertuples(index=False):
            cwt = (r.categories_within_threshold.split("+")
                   if isinstance(r.categories_within_threshold, str)
                   and r.categories_within_threshold else [])
            out.append(FileClassification(
                file_id=r.file_id,
                artifact_path=r.artifact_path,
                tool_name=r.tool_name,
                discovery_step=r.discovery_step,
                tool_category=None if pd.isna(r.tool_category) else r.tool_category,
                categories_within_threshold=cwt,
                signals_agree=bool(r.signals_agree),
                assigned_category=(None if pd.isna(r.assigned_category)
                                   else r.assigned_category),
            ))
        return out

    batch_results = []
    for full_repo_name, g in scored_pop.groupby("repo", sort=True):
        score = aggregate_repo_maturity(fc_list(g))
        org_name, repo_name = full_repo_name.split("/", 1)
        row = {
            "full_repo_name": full_repo_name,
            "org_name": org_name,
            "repo_name": repo_name,
            "level": score.overall_level,
            "label": score.overall_label,
            "confidence": score.confidence,
            "artifact_count": score.artifact_count,
            "tools": ", ".join(score.tools_detected),
            "n_tools": len(score.tools_detected),
            "l2_primary": score.level_evidence.get(2, {}).get("primary", 0),
            "l3_primary": score.level_evidence.get(3, {}).get("primary", 0),
            "l4_primary": score.level_evidence.get(4, {}).get("primary", 0),
            "l2_secondary": score.level_evidence.get(2, {}).get("secondary", 0),
            "l3_secondary": score.level_evidence.get(3, {}).get("secondary", 0),
            "l4_secondary": score.level_evidence.get(4, {}).get("secondary", 0),
        }
        for cat in CATEGORY_NAMES:
            row[f"cat_{cat}"] = score.category_counts.get(cat, 0)
        row["coherence_all_green"] = all(f.status == "green" for f in score.coherence_flags)
        batch_results.append(row)

    repo_scores_df = pd.DataFrame(batch_results)
    repo_scores_df.to_csv(CACHE_PATH, index=False)
    print(f"Cached to {CACHE_PATH}")

print(f"\nDataset: {len(repo_scores_df)} repositories scored")

In [ ]:
# --- Include frame repos filtered out by strict mode as Level 1 ---
# The sampling frame is the full 441-repo filtered population. Repos whose
# artifacts were ALL excluded by the strict whitelist (WL_STEPS) have no
# structured AI-artifact evidence under the strict definition, so they are
# included here as L1 (Ad Hoc) rather than dropped from the population.
INCLUDE_FILTERED_OUT_AS_L1 = True

if INCLUDE_FILTERED_OUT_AS_L1:
    frame_meta = pd.read_csv(DATA_DIR / "filtered_common_metadata.csv")  # FULL frame, no strict filter
    frame_repos = frame_meta[["repo_name", "org_name"]].drop_duplicates("repo_name")
    missing = frame_repos[~frame_repos["repo_name"].isin(repo_scores_df["full_repo_name"])]
    print(f"Sampling frame: {len(frame_repos)} repos; strict-scored: {len(repo_scores_df)}; "
          f"adding {len(missing)} filtered-out repos as L1")

    pad = pd.DataFrame({
        "full_repo_name": missing["repo_name"].values,
        "org_name": missing["org_name"].values,
        "repo_name": [rn.split("/", 1)[1] for rn in missing["repo_name"]],
        "level": 1,
        "label": MATURITY_LABELS[MaturityLevel.L1],
        "confidence": 1.0,
        "artifact_count": 0,
        "tools": "",
        "n_tools": 0,
    })
    for col in repo_scores_df.columns:
        if col not in pad.columns:
            pad[col] = True if col == "coherence_all_green" else 0

    repo_scores_df = (pd.concat([repo_scores_df, pad[repo_scores_df.columns]], ignore_index=True)
                        .sort_values("full_repo_name").reset_index(drop=True))
    print(f"Population for analysis: {len(repo_scores_df)} repos "
          f"({repo_scores_df['org_name'].nunique()} orgs)")
    print(repo_scores_df["level"].value_counts().sort_index().to_string())


In [ ]:
# Quick overview of the scored dataset
print("=== Dataset Overview ===")
print(f"Total repos: {len(repo_scores_df)}")
print(f"Orgs: {repo_scores_df['org_name'].nunique()}")
print(f"\nLevel distribution:")
print(repo_scores_df['level'].value_counts().sort_index())
print(f"\nArtifact count stats:")
print(repo_scores_df['artifact_count'].describe().round(1))

# Build binary category presence matrix (1 if repo has ≥1 artifact in that category)
cat_cols = [f"cat_{c}" for c in CATEGORY_NAMES]
cat_presence = (repo_scores_df[cat_cols] > 0).astype(int)
cat_presence.columns = CATEGORY_NAMES

# Add level info
cat_presence['level'] = repo_scores_df['level'].values
cat_presence['full_repo_name'] = repo_scores_df['full_repo_name'].values

print(f"\nCategory presence rates (% of repos with ≥1 artifact):")
for cat in CATEGORY_NAMES:
    pct = cat_presence[cat].mean() * 100
    print(f"  {cat:20s}: {pct:5.1f}%")

---

## RQ1c: Population Distribution — What Does the Maturity Landscape Look Like?

*Baseline descriptive analysis among repositories with detectable AI artifacts. Sets the stage for all subsequent questions.*

**Scope note**: This analysis covers the full 441-repo frame after QA filtering. Repos with no strict+W+ artifacts (231 of 441) are included as **L1** rather than excluded, so the question we answer is: *"Across the filtered population, how is maturity distributed?"*

Key questions:
- What proportion of repos are at each level (L1–L4)?
- Is the distribution unimodal, bimodal, or uniform?
- Does the distribution hold when restricted to repos with known AI tool associations?

In [ ]:
# --- Chart 1c.1: Maturity Level Distribution ---
level_counts = repo_scores_df['level'].value_counts().sort_index()
all_levels = [1, 2, 3, 4]
counts = [level_counts.get(lvl, 0) for lvl in all_levels]
total = sum(counts)

fig = go.Figure(go.Bar(
    x=[LEVEL_LABELS[lvl] for lvl in all_levels],
    y=counts,
    marker_color=[LEVEL_COLORS[lvl] for lvl in all_levels],
    text=[f"{c}<br>({c/total*100:.1f}%)" for c in counts],
    textposition='outside',
))
fig.update_layout(
    title=f"RQ1c: Maturity Level Distribution (N={total} repositories)",
    yaxis_title="Repository count",
    height=450, width=700,
    yaxis=dict(range=[0, max(counts) * 1.15]),
)
fig.show()
save_fig(fig, "rq1c_level_distribution")

# Statistical summary
print(f"Level distribution (N={total}):")
for lvl in all_levels:
    c = level_counts.get(lvl, 0)
    print(f"  L{lvl} {MATURITY_LABELS[MaturityLevel(lvl)]:25s}: {c:4d} ({c/total*100:.1f}%)")
print(f"\nMedian level: L{int(repo_scores_df['level'].median())}")
print(f"Mean level:   L{repo_scores_df['level'].mean():.2f}")

In [ ]:
# --- Chart 1c.2: Artifact Count Distribution by Level ---
fig = go.Figure()
for lvl in all_levels:
    subset = repo_scores_df[repo_scores_df['level'] == lvl]
    if len(subset) > 0:
        fig.add_trace(go.Box(
            y=subset['artifact_count'],
            name=LEVEL_LABELS[lvl],
            marker_color=LEVEL_COLORS[lvl],
            boxpoints='outliers',
        ))
fig.update_layout(
    title="Artifact Count Distribution by Maturity Level",
    yaxis_title="Number of artifacts per repo",
    height=450, width=700,
)
fig.show()

# Stats table
print("Artifact count by level:")
print(repo_scores_df.groupby('level')['artifact_count'].describe().round(1).to_string())

In [ ]:
# --- Chart 1c.3: Confidence Distribution by Level ---
fig = go.Figure()
for lvl in all_levels:
    subset = repo_scores_df[repo_scores_df['level'] == lvl]
    if len(subset) > 0:
        fig.add_trace(go.Histogram(
            x=subset['confidence'],
            name=LEVEL_LABELS[lvl],
            marker_color=LEVEL_COLORS[lvl],
            opacity=0.7,
            nbinsx=20,
        ))
fig.update_layout(
    title="Classification Confidence Distribution by Level",
    xaxis_title="Confidence score",
    yaxis_title="Repository count",
    barmode='overlay',
    height=400, width=700,
)
fig.show()

print("Confidence by level:")
print(repo_scores_df.groupby('level')['confidence'].describe().round(3).to_string())

In [ ]:
# --- Chart 1c.4: Category Prevalence Heatmap by Level ---
# For each level, compute % of repos that have each category
heatmap_data = []
for lvl in [2, 3, 4]:
    subset = cat_presence[cat_presence['level'] == lvl]
    if len(subset) > 0:
        rates = subset[CATEGORY_NAMES].mean() * 100
        heatmap_data.append(rates)
    else:
        heatmap_data.append(pd.Series({c: 0 for c in CATEGORY_NAMES}))

heatmap_df = pd.DataFrame(heatmap_data, index=["L2 repos", "L3 repos", "L4 repos"])

# Order categories by level then name
ordered_cats = sorted(CATEGORY_NAMES, key=lambda c: (CATEGORY_TO_LEVEL[c], c))

fig = go.Figure(go.Heatmap(
    z=heatmap_df[ordered_cats].values,
    x=[f"{c} (L{CATEGORY_TO_LEVEL[c]})" for c in ordered_cats],
    y=heatmap_df.index.tolist(),
    colorscale='YlOrRd',
    text=heatmap_df[ordered_cats].values.round(1),
    texttemplate='%{text}%',
    textfont={'size': 11},
    colorbar_title='% repos',
))
fig.update_layout(
    title="Category Prevalence by Maturity Level (% of repos with ≥1 artifact)",
    height=300, width=900,
)
fig.show()
save_fig(fig, "rq1c_category_heatmap")

### RQ1c Robustness: AI-Tools-Only Subset

The full dataset (441 repos) includes 224 repos with no known AI-tool association (padded L1 repos and repos where only "unknown"-tool artifacts were found by the fallback scanner). To verify that the maturity distribution reflects genuine AI adoption rather than generic markdown, we repeat the analysis on the **217-repo subset** where at least one known AI tool artifact (Claude Code, Cursor, Copilot, etc.) was detected.

In [ ]:
# --- Chart 1c.5: Robustness — Full Dataset vs AI-Tools Subset ---
ai_scores_df = repo_scores_df[repo_scores_df['full_repo_name'].isin(ai_tool_repos)]
non_ai_scores_df = repo_scores_df[~repo_scores_df['full_repo_name'].isin(ai_tool_repos)]

print(f"Full dataset:      {len(repo_scores_df)} repos")
print(f"AI-tools subset:   {len(ai_scores_df)} repos (≥1 known AI tool)")
print(f"Unknown-only repos: {len(non_ai_scores_df)} repos")

# Side-by-side level distributions
fig = make_subplots(rows=1, cols=2,
    subplot_titles=[
        f"Full Dataset (N={len(repo_scores_df)})",
        f"AI-Tools Subset (N={len(ai_scores_df)})"
    ])

for col, (label, sdf) in enumerate(
    [("Full", repo_scores_df), ("AI-Tools", ai_scores_df)], 1
):
    lc = sdf['level'].value_counts().sort_index()
    n = len(sdf)
    for lvl in [2, 3, 4]:
        c = lc.get(lvl, 0)
        fig.add_trace(go.Bar(
            x=[LEVEL_LABELS[lvl]],
            y=[c / n * 100],
            marker_color=LEVEL_COLORS[lvl],
            text=[f"{c}<br>({c/n*100:.1f}%)"],
            textposition='outside',
            showlegend=(col == 1),
            name=LEVEL_LABELS[lvl],
        ), row=1, col=col)

fig.update_layout(
    title="RQ1c Robustness: Maturity Distribution — Full vs AI-Tools Subset",
    height=450, width=900,
    barmode='group',
)
fig.update_yaxes(range=[0, 55], title_text="% of repos", row=1, col=1)
fig.update_yaxes(range=[0, 55], row=1, col=2)
fig.show()

# Statistical comparison
print("\nLevel distribution comparison:")
print(f"{'Level':<20} {'Full':>12} {'AI-Tools':>12} {'Unknown-Only':>14}")
print("-" * 60)
for lvl in [2, 3, 4]:
    fc = (repo_scores_df['level'] == lvl).sum()
    ac = (ai_scores_df['level'] == lvl).sum()
    nc = (non_ai_scores_df['level'] == lvl).sum()
    fn, an, nn = len(repo_scores_df), len(ai_scores_df), len(non_ai_scores_df)
    print(f"  L{lvl} {LEVEL_LABELS[lvl]:<15} {fc:4d} ({fc/fn*100:4.1f}%)  "
          f"{ac:4d} ({ac/an*100:4.1f}%)  {nc:4d} ({nc/nn*100:4.1f}%)")

# Chi-squared test: is the distribution significantly different?
from scipy.stats import chi2_contingency
contingency = pd.crosstab(
    repo_scores_df['full_repo_name'].isin(ai_tool_repos).map({True: 'AI-Tools', False: 'Unknown-Only'}),
    repo_scores_df['level']
)
chi2, p_chi2, dof, expected = chi2_contingency(contingency)
print(f"\nChi-squared test (AI-Tools vs Unknown-Only):")
print(f"  chi2={chi2:.2f}, dof={dof}, p={p_chi2:.4f}")
print(f"  → {'Significantly different' if p_chi2 < 0.05 else 'No significant difference'} at p<0.05")

---

## RQ1a: Category Co-occurrence — Is Adoption Structured as Cumulative Broadening?

*Do repos build outward from an L2 foundation, progressively adding higher-level categories?*

We test this with three complementary analyses:
1. **Jaccard co-occurrence matrix** — which categories appear together? Strong L2-L2 co-occurrence + weaker cross-level pairing suggests a foundation-first pattern.
2. **Category breadth vs. maturity level** — does breadth increase monotonically with level? A strong Spearman correlation means adoption is a gradient, not random sampling.
3. **Profile diversity** — how many unique category combinations exist? High diversity with a long tail argues against discrete archetypes and for a continuous maturity gradient.

Supplementary: hierarchical clustering on binary category presence vectors. Low silhouette scores would confirm that repos lie along a gradient rather than forming discrete profile clusters.

In [ ]:
# --- Chart 1a.1: Category Co-occurrence Matrix ---
# Jaccard similarity between categories (across repos)
n_cats = len(CATEGORY_NAMES)
jaccard_matrix = np.zeros((n_cats, n_cats))

for i, cat_i in enumerate(ordered_cats):
    for j, cat_j in enumerate(ordered_cats):
        set_i = set(cat_presence[cat_presence[cat_i] == 1].index)
        set_j = set(cat_presence[cat_presence[cat_j] == 1].index)
        if len(set_i | set_j) > 0:
            jaccard_matrix[i, j] = len(set_i & set_j) / len(set_i | set_j)
        else:
            jaccard_matrix[i, j] = 0

fig = go.Figure(go.Heatmap(
    z=jaccard_matrix,
    x=[f"{c} (L{CATEGORY_TO_LEVEL[c]})" for c in ordered_cats],
    y=[f"{c} (L{CATEGORY_TO_LEVEL[c]})" for c in ordered_cats],
    colorscale='Viridis',
    text=np.round(jaccard_matrix, 2),
    texttemplate='%{text}',
    textfont={'size': 9},
    colorbar_title='Jaccard',
))
fig.update_layout(
    title="Category Co-occurrence (Jaccard Similarity)",
    height=550, width=650,
)
fig.show()
save_fig(fig, "rq1a_cooccurrence_heatmap")

# Highlight strongest co-occurrences (off-diagonal)
print("Top 10 category co-occurrences (Jaccard similarity):")
pairs = []
for i in range(n_cats):
    for j in range(i + 1, n_cats):
        pairs.append((ordered_cats[i], ordered_cats[j], jaccard_matrix[i, j]))
pairs.sort(key=lambda x: -x[2])
for cat_i, cat_j, jac in pairs[:10]:
    print(f"  {cat_i:15s} + {cat_j:15s}: {jac:.3f}")

In [ ]:
# --- Chart 1a.2: Supplementary — Profile Clustering Attempt ---
# Cluster repos by their binary category presence vectors using hierarchical clustering.
# We expect weak clustering (low silhouette) because adoption follows a continuous
# maturity gradient rather than discrete archetypes.

presence_matrix = cat_presence[ordered_cats].values

# Compute pairwise Jaccard distance between repos
from sklearn.metrics import pairwise_distances
dist_matrix = pairwise_distances(presence_matrix, metric='jaccard')

# Hierarchical clustering
Z = linkage(squareform(dist_matrix, checks=False), method='ward')

# Find optimal number of clusters (2-8) using silhouette score
from sklearn.metrics import silhouette_score
sil_scores = {}
for k in range(2, 9):
    labels = fcluster(Z, k, criterion='maxclust')
    if len(set(labels)) > 1:
        sil_scores[k] = silhouette_score(dist_matrix, labels, metric='precomputed')
        
print("Silhouette scores by cluster count:")
for k, s in sil_scores.items():
    marker = " <<<" if s == max(sil_scores.values()) else ""
    print(f"  k={k}: {s:.3f}{marker}")

best_k = max(sil_scores, key=sil_scores.get)
best_sil = sil_scores[best_k]
print(f"\nBest k = {best_k} (silhouette = {best_sil:.3f})")
if best_sil < 0.30:
    print(f"\nInterpretation: silhouette < 0.30 indicates no strong discrete clusters.")
    print("This supports the cumulative broadening model — repos lie along a")
    print("continuous maturity gradient rather than forming distinct archetypes.")
else:
    print(f"\nInterpretation: silhouette >= 0.30 indicates discrete profile clusters.")
    print("With few unique category combinations, repos concentrate in a small set of")
    print("simple profiles; check whether clusters align with maturity levels below.")

fig = go.Figure(go.Bar(
    x=list(sil_scores.keys()),
    y=list(sil_scores.values()),
    marker_color=['#22c55e' if k == best_k else '#3b82f6' for k in sil_scores.keys()],
    text=[f"{s:.3f}" for s in sil_scores.values()],
    textposition='outside',
))
fig.update_layout(
    title="Supplementary: Profile Clustering (Silhouette Score)",
    xaxis_title="Number of clusters (k)",
    yaxis_title="Silhouette score",
    height=350, width=500,
    xaxis=dict(dtick=1),
    yaxis=dict(range=[0, max(sil_scores.values()) * 1.3]),
)
fig.show()

### Silhouette Null Model and Gradient Evidence

The silhouette score alone cannot distinguish a "structured gradient" from "noisy/unstructured data." We add a permutation null model and complementary tests to disambiguate.

In [ ]:
# --- Chart 1a.2a: Permutation Null Model for Silhouette ---
# Test whether the observed silhouette score reflects genuine structure or noise.
# Null model: shuffle each column of the binary presence matrix independently
# (preserves marginal category prevalences, breaks co-occurrence structure).

np.random.seed(42)
n_permutations = 1000
null_sils = []

for _ in range(n_permutations):
    perm_matrix = presence_matrix.copy()
    for col in range(perm_matrix.shape[1]):
        np.random.shuffle(perm_matrix[:, col])
    perm_dist = pairwise_distances(perm_matrix, metric='jaccard')
    Z_perm = linkage(squareform(perm_dist, checks=False), method='ward')
    labels_perm = fcluster(Z_perm, best_k, criterion='maxclust')
    if len(set(labels_perm)) > 1:
        null_sils.append(silhouette_score(perm_dist, labels_perm, metric='precomputed'))

null_sils = np.array(null_sils)
p_value = np.mean(null_sils >= best_sil)
null_mean = null_sils.mean()
null_95 = np.percentile(null_sils, 95)

print(f"=== Silhouette Permutation Test (n={n_permutations}) ===")
print(f"Observed silhouette (k={best_k}): {best_sil:.3f}")
print(f"Null distribution: mean={null_mean:.3f}, 95th pctile={null_95:.3f}")
print(f"p-value (fraction null >= observed): {p_value:.4f}")
print(f"\nInterpretation:")
if p_value < 0.05:
    print(f"  Observed silhouette is significantly above null (p={p_value:.4f}).")
    print(f"  The data has more structure than noise — rejects the 'noisy data' interpretation.")
else:
    print(f"  Observed silhouette is NOT significantly above null (p={p_value:.4f}).")
    print(f"  Cannot distinguish from noise based on silhouette alone.")

# Plot null distribution with observed value
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=null_sils, nbinsx=50, 
    marker_color='#94a3b8', name='Null distribution',
    opacity=0.7,
))
fig.add_vline(x=best_sil, line_dash="dash", line_color="#ef4444", line_width=2,
              annotation_text=f"Observed = {best_sil:.3f}", annotation_position="top right")
fig.add_vline(x=null_95, line_dash="dot", line_color="#f97316", line_width=1,
              annotation_text=f"95th pctile = {null_95:.3f}", annotation_position="top left")
fig.update_layout(
    title=f"Silhouette Permutation Test (k={best_k}, p={p_value:.4f})",
    xaxis_title="Silhouette score (null distribution)",
    yaxis_title="Count",
    height=400, width=700,
    showlegend=False,
)
save_fig(fig, "rq1a_silhouette_permutation")
fig.show()

In [ ]:
# --- Chart 1a.2b: Scored-population silhouette (padding-free robustness) ---
# The 441-frame silhouette above is partly mechanical: the padded L1 repos plus
# scored repos with empty profiles form one identical all-zero block that
# inflates both the observed score and the permutation null. Recompute the
# identical procedure on the scored population only (rq1_repo_scores.csv) —
# this is the value the paper cites in the Validation section (3.3.1).
scored_df = pd.read_csv(CACHE_PATH)
scored_presence = (scored_df[[f"cat_{c}" for c in CATEGORY_NAMES]] > 0).astype(int)
scored_presence.columns = CATEGORY_NAMES
sp_matrix = scored_presence[ordered_cats].values

sp_dist = pairwise_distances(sp_matrix, metric='jaccard')
Z_sp = linkage(squareform(sp_dist, checks=False), method='ward')
sp_sils = {}
for k in range(2, 9):
    labels = fcluster(Z_sp, k, criterion='maxclust')
    if len(set(labels)) > 1:
        sp_sils[k] = silhouette_score(sp_dist, labels, metric='precomputed')
sp_best_k = max(sp_sils, key=sp_sils.get)
sp_best = sp_sils[sp_best_k]

np.random.seed(42)
sp_nulls = []
for _ in range(1000):
    perm = sp_matrix.copy()
    for col in range(perm.shape[1]):
        np.random.shuffle(perm[:, col])
    perm_dist = pairwise_distances(perm, metric='jaccard')
    lp = fcluster(linkage(squareform(perm_dist, checks=False), method='ward'), sp_best_k, criterion='maxclust')
    if len(set(lp)) > 1:
        sp_nulls.append(silhouette_score(perm_dist, lp, metric='precomputed'))
sp_nulls = np.array(sp_nulls)

print(f"=== Scored-population silhouette (n={len(scored_df)} repos, no padding) ===")
print(f"Best k = {sp_best_k}, observed silhouette = {sp_best:.3f}")
print(f"Null (column-shuffle, n=1000): mean={sp_nulls.mean():.3f}, 95th pctile={np.percentile(sp_nulls, 95):.3f}")
print(f"p-value (fraction null >= observed): {np.mean(sp_nulls >= sp_best):.4f}")
print(f"Unique category combinations: {scored_presence[ordered_cats].apply(tuple, axis=1).nunique()}")
print(f"Top-15 combination coverage: {scored_presence[ordered_cats].apply(tuple, axis=1).value_counts().head(15).sum()/len(scored_df)*100:.1f}%")


In [ ]:
# --- Chart 1a.2b: Hopkins Statistic, PCoA, and Unimodality Test ---
# Hopkins statistic: tests whether data has non-random spatial structure (H > 0.5).
# PCoA: project Jaccard distances into 2D to visualize the gradient.
# Dip test: test whether the first principal coordinate is unimodal (gradient)
# or multimodal (hidden clusters).

from sklearn.neighbors import NearestNeighbors

# --- Hopkins statistic ---
# Sample m random points, compare distance to nearest real neighbor
# vs distance to nearest neighbor of a random uniform point.
np.random.seed(42)
m = min(50, len(presence_matrix) // 5)
n_samples = len(presence_matrix)

# Sample m data points
sample_idx = np.random.choice(n_samples, m, replace=False)
remaining_idx = np.setdiff1d(np.arange(n_samples), sample_idx)

# For each sampled point, find distance to nearest OTHER real point
nn = NearestNeighbors(n_neighbors=2, metric='jaccard')
nn.fit(presence_matrix)
dists_real, _ = nn.kneighbors(presence_matrix[sample_idx])
w = dists_real[:, 1]  # skip self (dist=0)

# Generate uniform random binary vectors with same marginal prevalences
marginals = presence_matrix.mean(axis=0)
random_points = np.array([
    (np.random.random(presence_matrix.shape[1]) < marginals).astype(int)
    for _ in range(m)
])
dists_random, _ = nn.kneighbors(random_points)
u = dists_random[:, 0]

H = u.sum() / (u.sum() + w.sum())

print(f"=== Hopkins Statistic ===")
print(f"H = {H:.3f} (m={m})")
print(f"  H ≈ 0.5 → random/uniform (no clustering tendency)")
print(f"  H > 0.5 → non-random structure exists")
print(f"  H > 0.75 → strong clustering tendency")
if H > 0.5:
    print(f"  Result: Data has non-random spatial structure (H={H:.3f} > 0.5)")
else:
    print(f"  Result: Data shows no clustering tendency (H={H:.3f} ≈ 0.5)")

# --- PCoA (Classical MDS on Jaccard distances) ---
from sklearn.manifold import MDS
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
coords = mds.fit_transform(dist_matrix)

# Color by level
levels = cat_presence['level'].values
fig = go.Figure()
for lvl in [2, 3, 4]:
    mask = levels == lvl
    fig.add_trace(go.Scatter(
        x=coords[mask, 0], y=coords[mask, 1],
        mode='markers', name=f'L{lvl}',
        marker=dict(color=LEVEL_COLORS[lvl], size=5, opacity=0.6),
    ))
fig.update_layout(
    title="PCoA of Jaccard Distances (colored by maturity level)",
    xaxis_title="PC1", yaxis_title="PC2",
    height=500, width=600, legend_title="Level",
)
save_fig(fig, "rq1a_pcoa_gradient")
fig.show()

# --- Dip test on first principal coordinate ---
# Hartigan's dip test: H0 = unimodal. Low p = reject unimodality.
pc1 = coords[:, 0]
try:
    from diptest import diptest as dip_test
    dip_stat, dip_p = dip_test(pc1)
    print(f"\n=== Hartigan Dip Test on PC1 ===")
    print(f"Dip statistic: {dip_stat:.4f}, p-value: {dip_p:.4f}")
    if dip_p > 0.05:
        print(f"  Cannot reject unimodality (p={dip_p:.3f}) → consistent with gradient")
    else:
        print(f"  Reject unimodality (p={dip_p:.3f}) → evidence of multimodal structure")
except ImportError:
    print("\n(diptest package not available — install with: pip install diptest)")
    print("Skipping Hartigan dip test. Visual PCoA inspection suggests gradient structure.")

In [ ]:
# --- Chart 1a.3: Supplementary — Profile Characterization ---
# Although discrete profiles are not the primary finding, we characterize
# the clusters to show they align with the maturity gradient (L2→L3→L4)
# rather than representing orthogonal specializations.
cluster_labels = fcluster(Z, best_k, criterion='maxclust')
cat_presence['profile'] = cluster_labels

# Characterize each profile: average category presence and dominant maturity level
profile_summary = []
for p in sorted(cat_presence['profile'].unique()):
    subset = cat_presence[cat_presence['profile'] == p]
    n = len(subset)
    level_mode = subset['level'].mode().iloc[0]
    level_dist = subset['level'].value_counts().to_dict()
    avg_cats = subset[ordered_cats].mean()
    
    dominant = [c for c in ordered_cats if avg_cats[c] > 0.5]
    present = [c for c in ordered_cats if avg_cats[c] > 0.25]
    
    profile_summary.append({
        'profile': p,
        'n_repos': n,
        'pct': n / len(cat_presence) * 100,
        'dominant_level': f"L{level_mode}",
        'level_dist': level_dist,
        'dominant_cats': dominant,
        'present_cats': present,
    })

print(f"=== {best_k} Artifact Profiles (Supplementary) ===\n")
for ps in profile_summary:
    print(f"Profile {ps['profile']} (n={ps['n_repos']}, {ps['pct']:.1f}%):")
    print(f"  Dominant level: {ps['dominant_level']}")
    print(f"  Level distribution: {ps['level_dist']}")
    print(f"  Dominant categories (>50%): {ps['dominant_cats'] or 'none'}")
    print(f"  Present categories (>25%):  {ps['present_cats'] or 'none'}")
    print()

# Note: profiles align with maturity levels, not orthogonal specializations
level_alignment = sum(1 for ps in profile_summary
    if len(set(ps['level_dist'].keys())) <= 2)
print(f"Profile-level alignment: {level_alignment}/{len(profile_summary)} profiles "
      f"map to ≤2 adjacent maturity levels.")
print(f"This confirms that forced clusters recover the maturity gradient,")
print(f"not independent specialization dimensions.")

# Heatmap of profiles × categories
profile_heatmap = []
profile_labels = []
for p in sorted(cat_presence['profile'].unique()):
    subset = cat_presence[cat_presence['profile'] == p]
    n = len(subset)
    profile_heatmap.append((subset[ordered_cats].mean() * 100).values)
    lvl = subset['level'].mode().iloc[0]
    profile_labels.append(f"Profile {p} (n={n}, L{lvl})")

fig = go.Figure(go.Heatmap(
    z=profile_heatmap,
    x=[f"{c} (L{CATEGORY_TO_LEVEL[c]})" for c in ordered_cats],
    y=profile_labels,
    colorscale='YlOrRd',
    text=np.round(profile_heatmap, 1),
    texttemplate='%{text}%',
    textfont={'size': 10},
    colorbar_title='% repos',
))
fig.update_layout(
    title=f"Supplementary: Artifact Profiles (k={best_k} forced clusters)",
    height=max(250, best_k * 60 + 100), width=900,
)
fig.show()

In [ ]:
# --- Chart 1a.4: Most Common Category Combinations (UpSet-style) ---
# For each repo, build a frozenset of present categories, then count frequencies

repo_profiles = []
for idx, row in cat_presence.iterrows():
    cats = frozenset(c for c in CATEGORY_NAMES if row[c] == 1)
    repo_profiles.append(cats)

profile_counts = Counter(repo_profiles)
top_profiles = profile_counts.most_common(15)

print(f"Unique category combinations: {len(profile_counts)}")
print(f"\nTop 15 most common artifact profiles:")
print(f"{'Rank':>4}  {'Count':>5}  {'%':>6}  Categories")
print("-" * 70)
cumulative = 0
for i, (cats, count) in enumerate(top_profiles, 1):
    cumulative += count
    cat_str = " + ".join(sorted(cats)) if cats else "(empty)"
    print(f"{i:4d}  {count:5d}  {count/len(cat_presence)*100:5.1f}%  {cat_str}")
print(f"\nTop 15 cover {cumulative}/{len(cat_presence)} repos ({cumulative/len(cat_presence)*100:.1f}%)")

# Bar chart of top profiles
labels = []
counts_list = []
for cats, count in top_profiles:
    label = " + ".join(sorted(cats)) if cats else "(empty)"
    # Truncate long labels
    if len(label) > 50:
        label = label[:47] + "..."
    labels.append(label)
    counts_list.append(count)

fig = go.Figure(go.Bar(
    y=labels[::-1],
    x=counts_list[::-1],
    orientation='h',
    marker_color='#3b82f6',
    text=counts_list[::-1],
    textposition='outside',
))
fig.update_layout(
    title="Most Common Category Combinations (Top 15)",
    xaxis_title="Repository count",
    height=max(400, len(labels) * 30 + 100), width=900,
    margin=dict(l=350),
)
fig.show()

In [ ]:
# --- Chart 1a.5: Category Breadth Distribution ---
# How many distinct categories does each repo have?
cat_presence['n_categories'] = cat_presence[CATEGORY_NAMES].sum(axis=1)

fig = go.Figure(go.Histogram(
    x=cat_presence['n_categories'],
    marker_color='#3b82f6',
    nbinsx=10,
))
fig.update_layout(
    title="Category Breadth: How Many Categories Per Repo?",
    xaxis_title="Number of distinct categories present",
    yaxis_title="Repository count",
    height=400, width=600,
    xaxis=dict(dtick=1),
)
fig.show()
save_fig(fig, "rq1a_breadth_by_level")

print(f"Category breadth stats:")
print(cat_presence['n_categories'].describe().round(2))
print(f"\nBy level:")
for lvl in [2, 3, 4]:
    subset = cat_presence[cat_presence['level'] == lvl]
    print(f"  L{lvl}: mean={subset['n_categories'].mean():.1f}, "
          f"median={subset['n_categories'].median():.0f}, "
          f"range=[{subset['n_categories'].min()}-{subset['n_categories'].max()}]")

# Test: does category breadth increase with level? (Spearman correlation)
rho, p = stats.spearmanr(cat_presence['level'], cat_presence['n_categories'])
print(f"\nSpearman correlation (level vs breadth): rho={rho:.3f}, p={p:.2e}")
print(f"  → {'Significant' if p < 0.05 else 'Not significant'} at p<0.05")

In [ ]:
# --- Chart 1a.6: Robustness — Breadth-Level Correlation on AI-Tools Subset ---
ai_cat_presence = cat_presence[cat_presence['full_repo_name'].isin(ai_tool_repos)]

rho_ai, p_ai = stats.spearmanr(ai_cat_presence['level'], ai_cat_presence['n_categories'])

print(f"=== Breadth-Level Correlation Robustness ===\n")
print(f"{'Dataset':<25} {'N':>5} {'Spearman rho':>14} {'p-value':>14}")
print("-" * 60)
print(f"{'Full dataset':<25} {len(cat_presence):>5} {rho:>14.3f} {p:>14.2e}")
print(f"{'AI-tools subset':<25} {len(ai_cat_presence):>5} {rho_ai:>14.3f} {p_ai:>14.2e}")

print(f"\nBreadth by level (AI-tools subset, N={len(ai_cat_presence)}):")
for lvl in [2, 3, 4]:
    subset = ai_cat_presence[ai_cat_presence['level'] == lvl]
    if len(subset) > 0:
        print(f"  L{lvl}: mean={subset['n_categories'].mean():.1f}, "
              f"median={subset['n_categories'].median():.0f}, "
              f"n={len(subset)}")

print(f"\nConclusion: The breadth-level correlation {'holds' if p_ai < 0.05 else 'does NOT hold'} "
      f"on the AI-tools subset (rho={rho_ai:.3f}), confirming that cumulative ")
print(f"broadening is not an artifact of generic markdown inclusion.")

---

## RQ1b: Is Maturity Cumulative? (Central Validity Test)

*Do L4 repos contain L2+L3 artifacts? Do L3 repos contain L2? Or are the levels a menu, not a ladder?*

This is the **central validity test** for the AIME framework. If L4 repos routinely lack L2 artifacts, the "maturity" framing is wrong — it's specialization, not progression.

**Hypothesis**: If maturity is cumulative (a Guttman scale), then:
- All L3 repos should have L2 evidence
- All L4 repos should have both L2 and L3 evidence
- Violations (e.g., L4 without L2) indicate the model is not a true progression

In [ ]:
# --- Analysis 1b.1: Cumulative Evidence Test ---
# For each repo, check whether it has primary evidence at each level
df = repo_scores_df.copy()
df['has_l2'] = df['l2_primary'] > 0
df['has_l3'] = df['l3_primary'] > 0
df['has_l4'] = df['l4_primary'] > 0

# Build the cumulative pattern for each repo
def cumulative_pattern(row):
    parts = []
    if row['has_l2']: parts.append('L2')
    if row['has_l3']: parts.append('L3')
    if row['has_l4']: parts.append('L4')
    return '+'.join(parts) if parts else 'none'

df['evidence_pattern'] = df.apply(cumulative_pattern, axis=1)

# Crosstab: level vs evidence pattern
print("=== Cumulative Maturity Test ===\n")
print("Q: Do higher-level repos contain lower-level evidence?\n")

for lvl in [3, 4]:
    subset = df[df['level'] == lvl]
    n = len(subset)
    if n == 0:
        continue
    print(f"L{lvl} repos (n={n}):")
    if lvl == 3:
        with_l2 = subset['has_l2'].sum()
        print(f"  Has L2 evidence: {with_l2}/{n} ({with_l2/n*100:.1f}%)")
        print(f"  Missing L2:      {n - with_l2}/{n} ({(n-with_l2)/n*100:.1f}%)")
    elif lvl == 4:
        with_l2 = subset['has_l2'].sum()
        with_l3 = subset['has_l3'].sum()
        with_both = ((subset['has_l2']) & (subset['has_l3'])).sum()
        print(f"  Has L2 evidence: {with_l2}/{n} ({with_l2/n*100:.1f}%)")
        print(f"  Has L3 evidence: {with_l3}/{n} ({with_l3/n*100:.1f}%)")
        print(f"  Has both L2+L3:  {with_both}/{n} ({with_both/n*100:.1f}%)")
        print(f"  Missing L2 or L3: {n - with_both}/{n} ({(n-with_both)/n*100:.1f}%)")
    print()

# Including secondary (embedded) evidence
print("--- With secondary (embedded) evidence included ---\n")
df['has_l2_any'] = (df['l2_primary'] + df['l2_secondary']) > 0
df['has_l3_any'] = (df['l3_primary'] + df['l3_secondary']) > 0

for lvl in [3, 4]:
    subset = df[df['level'] == lvl]
    n = len(subset)
    if n == 0:
        continue
    print(f"L{lvl} repos (n={n}):")
    if lvl == 3:
        with_l2 = subset['has_l2_any'].sum()
        print(f"  Has L2 evidence (primary+secondary): {with_l2}/{n} ({with_l2/n*100:.1f}%)")
    elif lvl == 4:
        with_l2 = subset['has_l2_any'].sum()
        with_l3 = subset['has_l3_any'].sum()
        with_both = ((subset['has_l2_any']) & (subset['has_l3_any'])).sum()
        print(f"  Has L2 evidence: {with_l2}/{n} ({with_l2/n*100:.1f}%)")
        print(f"  Has L3 evidence: {with_l3}/{n} ({with_l3/n*100:.1f}%)")
        print(f"  Has both L2+L3:  {with_both}/{n} ({with_both/n*100:.1f}%)")
    print()

In [ ]:
# --- Chart 1b.2: Evidence Pattern Stacked Bar ---
# Visualize what evidence each level contains

fig = make_subplots(rows=1, cols=3, subplot_titles=["L2 Repos", "L3 Repos", "L4 Repos"])

for col, lvl in enumerate([2, 3, 4], 1):
    subset = df[df['level'] == lvl]
    n = len(subset)
    if n == 0:
        continue
    
    # Compute % with evidence at each level
    evidence = {
        'L2 primary': (subset['l2_primary'] > 0).mean() * 100,
        'L3 primary': (subset['l3_primary'] > 0).mean() * 100,
        'L4 primary': (subset['l4_primary'] > 0).mean() * 100,
    }
    
    fig.add_trace(go.Bar(
        x=list(evidence.keys()),
        y=list(evidence.values()),
        marker_color=[LEVEL_COLORS[2], LEVEL_COLORS[3], LEVEL_COLORS[4]],
        text=[f"{v:.0f}%" for v in evidence.values()],
        textposition='outside',
        showlegend=False,
    ), row=1, col=col)

fig.update_layout(
    title="Cumulative Evidence: % of Repos with Primary Evidence at Each Level",
    height=400, width=900,
)
fig.update_yaxes(range=[0, 110], title_text="% of repos" if col == 1 else None)
fig.show()
save_fig(fig, "rq1b_cumulative_evidence")

In [ ]:
# --- Chart 1b.3: Guttman Scalogram ---
# A Guttman scale test: if items are ordered by difficulty, response patterns
# should be cumulative (1,1,1,0 but never 1,0,1,0).
# We test this with the Coefficient of Reproducibility (CR) and Scalability (CS).

# Binary matrix: repos x levels (has primary evidence at L2, L3, L4)
guttman_matrix = df[['has_l2', 'has_l3', 'has_l4']].astype(int).values
n_repos = len(guttman_matrix)

# Count error patterns (violations of cumulative ordering)
# Valid patterns for a 3-item Guttman scale: 000, 100, 110, 111
valid_patterns = {(0,0,0), (1,0,0), (1,1,0), (1,1,1)}
errors = 0
error_patterns = Counter()
for row in guttman_matrix:
    pattern = tuple(row)
    if pattern not in valid_patterns:
        errors += 1
        error_patterns[pattern] += 1

total_responses = n_repos * 3  # 3 items per repo
# Count individual cell errors (minimum flips to make pattern valid)
cell_errors = 0
for row in guttman_matrix:
    pattern = tuple(row)
    if pattern not in valid_patterns:
        # Find closest valid pattern
        min_dist = min(sum(a != b for a, b in zip(pattern, vp)) for vp in valid_patterns)
        cell_errors += min_dist

CR = 1 - (cell_errors / total_responses)
# Minimum Marginal Reproducibility
marginals = guttman_matrix.mean(axis=0)
MMR = sum(max(p, 1-p) for p in marginals) / 3
CS = (CR - MMR) / (1 - MMR) if MMR < 1 else 0

print("=== Guttman Scale Analysis ===\n")
print(f"Coefficient of Reproducibility (CR): {CR:.3f}")
print(f"  Threshold for acceptable scale: CR >= 0.90")
print(f"  Result: {'PASS' if CR >= 0.90 else 'FAIL'}")
print(f"\nCoefficient of Scalability (CS): {CS:.3f}")
print(f"  Threshold for acceptable scale: CS >= 0.60")
print(f"  Result: {'PASS' if CS >= 0.60 else 'FAIL'}")
print(f"\nError patterns (non-cumulative):")
print(f"  Total repos with violations: {errors}/{n_repos} ({errors/n_repos*100:.1f}%)")
for pattern, count in error_patterns.most_common():
    labels = ['L2', 'L3', 'L4']
    desc = ', '.join(f"{l}={'yes' if v else 'no'}" for l, v in zip(labels, pattern))
    print(f"  ({desc}): {count} repos")

# Visualize the scalogram
pattern_labels = []
pattern_counts = []
pattern_colors = []
for pattern in [(1,1,1), (1,1,0), (1,0,0), (0,0,0)] + sorted(error_patterns.keys(), key=lambda x: -error_patterns[x]):
    count = sum(1 for row in guttman_matrix if tuple(row) == pattern)
    if count > 0 and pattern not in [p for p, _ in zip(pattern_labels, pattern_counts)]:
        labels = ['L2', 'L3', 'L4']
        desc = ' '.join(f"{'['+l+']' if v else ' -- '}" for l, v in zip(labels, pattern))
        pattern_labels.append(desc)
        pattern_counts.append(count)
        is_valid = pattern in valid_patterns
        pattern_colors.append('#22c55e' if is_valid else '#ef4444')

fig = go.Figure(go.Bar(
    y=pattern_labels[::-1],
    x=pattern_counts[::-1],
    orientation='h',
    marker_color=pattern_colors[::-1],
    text=pattern_counts[::-1],
    textposition='outside',
))
fig.update_layout(
    title=f"Guttman Scalogram: Evidence Patterns (CR={CR:.3f})",
    xaxis_title="Repository count",
    height=max(300, len(pattern_labels) * 35 + 100), width=700,
    margin=dict(l=200),
)
fig.show()
save_fig(fig, "rq1b_guttman_scalogram")

In [ ]:
# --- Chart 1b.4: Evidence Strength by Level (Stacked Area) ---
# For repos at each level, how much evidence do they have at each level?
# This shows whether higher-level repos have PROPORTIONALLY more lower-level evidence

fig = go.Figure()
for evidence_level, color in [(2, LEVEL_COLORS[2]), (3, LEVEL_COLORS[3]), (4, LEVEL_COLORS[4])]:
    means = []
    for repo_level in [2, 3, 4]:
        subset = df[df['level'] == repo_level]
        if len(subset) > 0:
            means.append(subset[f'l{evidence_level}_primary'].mean())
        else:
            means.append(0)
    
    fig.add_trace(go.Bar(
        x=[LEVEL_LABELS[l] for l in [2, 3, 4]],
        y=means,
        name=f"L{evidence_level} evidence",
        marker_color=color,
        text=[f"{m:.1f}" for m in means],
        textposition='auto',
    ))

fig.update_layout(
    title="Average Primary Evidence Count by Repo Level",
    xaxis_title="Repository maturity level",
    yaxis_title="Mean primary artifact count",
    barmode='group',
    height=450, width=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

In [ ]:
# --- Analysis 1b.5: Robustness — Guttman Scale on AI-Tools Subset ---
df_ai = df[df['full_repo_name'].isin(ai_tool_repos)]

guttman_ai = df_ai[['has_l2', 'has_l3', 'has_l4']].astype(int).values
n_ai = len(guttman_ai)

valid_patterns = {(0,0,0), (1,0,0), (1,1,0), (1,1,1)}
cell_errors_ai = 0
errors_ai = 0
for row in guttman_ai:
    pattern = tuple(row)
    if pattern not in valid_patterns:
        errors_ai += 1
        min_dist = min(sum(a != b for a, b in zip(pattern, vp)) for vp in valid_patterns)
        cell_errors_ai += min_dist

total_responses_ai = n_ai * 3
CR_ai = 1 - (cell_errors_ai / total_responses_ai)
marginals_ai = guttman_ai.mean(axis=0)
MMR_ai = sum(max(p, 1-p) for p in marginals_ai) / 3
CS_ai = (CR_ai - MMR_ai) / (1 - MMR_ai) if MMR_ai < 1 else 0

print("=== Guttman Scale Robustness: AI-Tools Subset ===\n")
print(f"{'Metric':<35} {'Full (N={})'.format(len(df)):>16} {'AI-Tools (N={})'.format(n_ai):>16}")
print("-" * 70)
print(f"{'Coefficient of Reproducibility':<35} {CR:>16.3f} {CR_ai:>16.3f}")
print(f"{'Coefficient of Scalability':<35} {CS:>16.3f} {CS_ai:>16.3f}")
print(f"{'Violation rate':<35} {errors/len(df)*100:>15.1f}% {errors_ai/n_ai*100:>15.1f}%")

print(f"\nBoth CR and CS {'PASS' if CR_ai >= 0.90 and CS_ai >= 0.60 else 'FAIL'} "
      f"thresholds on the AI-tools subset.")

# Cumulative evidence on AI-tools subset
print(f"\nCumulative evidence (AI-tools subset):")
l3_ai = df_ai[df_ai['level'] == 3]
l4_ai = df_ai[df_ai['level'] == 4]
if len(l3_ai) > 0:
    print(f"  L3 repos with L2 foundation: {l3_ai['has_l2'].mean()*100:.1f}% (n={len(l3_ai)})")
if len(l4_ai) > 0:
    both = ((l4_ai['has_l2']) & (l4_ai['has_l3'])).mean() * 100
    print(f"  L4 repos with L2+L3 foundation: {both:.1f}% (n={len(l4_ai)})")

### Blind Validation: Is the A Priori Hierarchy Empirically Grounded?

The Guttman scale test (above) validates cumulative ordering, but the category-to-level mapping is defined a priori. These tests check whether the specific mapping is special — i.e., whether it fits the data better than random alternatives.

In [ ]:
# --- Analysis 1b.6: Permutation Test — Is the A Priori Hierarchy Special? ---
# Test whether the specific category-to-level mapping produces higher Guttman CR/CS
# than random assignments. This addresses the circularity concern: if ANY mapping
# of 9 categories into (4, 3, 2) level groups achieves high CR/CS, then the a priori
# mapping is not validated by the Guttman test. If only the a priori mapping (and
# nearby permutations) achieve high CR/CS, the hierarchy is empirically grounded.

np.random.seed(42)
n_perms = 10000
cats = list(CATEGORY_NAMES)

# A priori mapping
a_priori_l2 = {c for c, l in CATEGORY_TO_LEVEL.items() if l == MaturityLevel.L2}
a_priori_l3 = {c for c, l in CATEGORY_TO_LEVEL.items() if l == MaturityLevel.L3}
a_priori_l4 = {c for c, l in CATEGORY_TO_LEVEL.items() if l == MaturityLevel.L4}
print(f"A priori mapping: L2={sorted(a_priori_l2)}, L3={sorted(a_priori_l3)}, L4={sorted(a_priori_l4)}")

# Reusable Guttman computation
def compute_guttman_cr_cs(has_l2_arr, has_l3_arr, has_l4_arr):
    matrix = np.column_stack([has_l2_arr, has_l3_arr, has_l4_arr])
    n = len(matrix)
    valid = {(0,0,0), (1,0,0), (1,1,0), (1,1,1)}
    cell_errs = 0
    for row in matrix:
        pattern = tuple(row)
        if pattern not in valid:
            min_dist = min(sum(a != b for a, b in zip(pattern, vp)) for vp in valid)
            cell_errs += min_dist
    total = n * 3
    cr = 1 - (cell_errs / total)
    marginals = matrix.mean(axis=0)
    mmr = sum(max(p, 1-p) for p in marginals) / 3
    cs = (cr - mmr) / (1 - mmr) if mmr < 1 else 0
    return cr, cs

# Observed CR/CS (should match RQ1b)
obs_cr, obs_cs = compute_guttman_cr_cs(
    df['has_l2'].astype(int).values,
    df['has_l3'].astype(int).values,
    df['has_l4'].astype(int).values,
)
print(f"\nObserved CR={obs_cr:.3f}, CS={obs_cs:.3f}")

# Permutation sweep: random mappings of 9 categories to (4, 3, 2) groups
perm_crs, perm_css = [], []
for _ in range(n_perms):
    shuffled = cats.copy()
    np.random.shuffle(shuffled)
    perm_l2 = set(shuffled[:4])
    perm_l3 = set(shuffled[4:7])
    perm_l4 = set(shuffled[7:])
    
    perm_has_l2 = cat_presence[list(perm_l2)].any(axis=1).astype(int).values
    perm_has_l3 = cat_presence[list(perm_l3)].any(axis=1).astype(int).values
    perm_has_l4 = cat_presence[list(perm_l4)].any(axis=1).astype(int).values
    
    cr, cs = compute_guttman_cr_cs(perm_has_l2, perm_has_l3, perm_has_l4)
    perm_crs.append(cr)
    perm_css.append(cs)

perm_crs = np.array(perm_crs)
perm_css = np.array(perm_css)
cr_pctile = np.mean(perm_crs >= obs_cr) * 100
cs_pctile = np.mean(perm_css >= obs_cs) * 100
cr_p = np.mean(perm_crs >= obs_cr)
cs_p = np.mean(perm_css >= obs_cs)

print(f"\n=== Permutation Test ({n_perms:,} random category-to-level mappings) ===")
print(f"Observed CR={obs_cr:.3f} — percentile rank: {100-cr_pctile:.1f}th (p={cr_p:.4f})")
print(f"Observed CS={obs_cs:.3f} — percentile rank: {100-cs_pctile:.1f}th (p={cs_p:.4f})")
print(f"Null CR: mean={perm_crs.mean():.3f}, max={perm_crs.max():.3f}")
print(f"Null CS: mean={perm_css.mean():.3f}, max={perm_css.max():.3f}")

# Plot
from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Coefficient of Reproducibility (CR)", f"Coefficient of Scalability (CS)"
])
fig.add_trace(go.Histogram(x=perm_crs, nbinsx=50, marker_color='#94a3b8', opacity=0.7,
                            name='Null (random mappings)'), row=1, col=1)
fig.add_vline(x=obs_cr, line_dash="dash", line_color="#ef4444", line_width=2,
              annotation_text=f"Observed={obs_cr:.3f}", row=1, col=1)

fig.add_trace(go.Histogram(x=perm_css, nbinsx=50, marker_color='#94a3b8', opacity=0.7,
                            name='Null (random mappings)', showlegend=False), row=1, col=2)
fig.add_vline(x=obs_cs, line_dash="dash", line_color="#ef4444", line_width=2,
              annotation_text=f"Observed={obs_cs:.3f}", row=1, col=2)

fig.update_layout(
    title=f"Guttman Permutation Test: Is the A Priori Hierarchy Special? (n={n_perms:,})",
    height=400, width=900, showlegend=False,
)
save_fig(fig, "rq1b_guttman_permutation")
fig.show()

In [ ]:
# --- Analysis 1b.7: Data-Driven Hierarchy Recovery ---
# Can the L2 < L3 < L4 ordering be recovered from prevalence alone, blind to labels?
# In a Guttman model, "easier" items (lower-level) should have higher prevalence.

# 1. Category prevalence
prevalences = cat_presence[list(CATEGORY_NAMES)].mean().sort_values(ascending=False)
print("=== Category Prevalence (% of repos) ===")
for cat, prev in prevalences.items():
    a_priori = int(CATEGORY_TO_LEVEL[cat])
    print(f"  {cat:20s}: {prev*100:5.1f}%  (a priori: L{a_priori})")

# 2. Kendall tau: prevalence rank vs a priori level
prev_rank = prevalences.rank(ascending=False)  # higher prevalence = lower rank number
a_priori_levels = pd.Series({c: int(CATEGORY_TO_LEVEL[c]) for c in CATEGORY_NAMES})
tau, tau_p = stats.kendalltau(
    [prev_rank[c] for c in CATEGORY_NAMES],
    [a_priori_levels[c] for c in CATEGORY_NAMES]
)
print(f"\n=== Kendall Tau: Prevalence Rank vs A Priori Level ===")
print(f"tau = {tau:.3f}, p = {tau_p:.4f}")
print(f"Positive tau means higher-prevalence categories are assigned to lower levels (as expected).")

# 3. Conditional probability matrix: P(category_i present | category_j present)
# For a valid scale, P(easier | harder) should be near 1.0
print(f"\n=== Conditional Probabilities: P(row | col) ===")
# Order categories by prevalence (easiest first)
ordered_by_prev = prevalences.index.tolist()
cond_matrix = pd.DataFrame(index=ordered_by_prev, columns=ordered_by_prev, dtype=float)
for cat_j in ordered_by_prev:
    has_j = cat_presence[cat_j] == 1
    n_j = has_j.sum()
    for cat_i in ordered_by_prev:
        if n_j > 0:
            cond_matrix.loc[cat_i, cat_j] = cat_presence.loc[has_j, cat_i].mean()
        else:
            cond_matrix.loc[cat_i, cat_j] = 0

# Display: for each "harder" category (lower prevalence), show P(easier categories | harder)
print("\nP(easier | harder) — should be close to 1.0 for valid cumulative scale:")
for i, harder in enumerate(ordered_by_prev):
    for easier in ordered_by_prev[:i]:
        p = cond_matrix.loc[easier, harder]
        lvl_easier = int(CATEGORY_TO_LEVEL[easier])
        lvl_harder = int(CATEGORY_TO_LEVEL[harder])
        if lvl_harder > lvl_easier:
            marker = " ✓" if p > 0.80 else " ✗"
            print(f"  P({easier} | {harder}) = {p:.3f}{marker}")

# 4. Loevinger H coefficient (scalability per pair)
# H_ij = 1 - (observed errors / expected errors under independence)
print(f"\n=== Loevinger H Coefficients ===")
h_values = []
for i, cat_i in enumerate(ordered_by_prev):
    for j, cat_j in enumerate(ordered_by_prev):
        if i >= j:
            continue
        pi = cat_presence[cat_i].mean()
        pj = cat_presence[cat_j].mean()
        if pi >= pj:
            easier, harder = cat_i, cat_j
            pe, ph = pi, pj
        else:
            easier, harder = cat_j, cat_i
            pe, ph = pj, pi
        # Expected errors under independence: P(harder=1, easier=0) = ph * (1 - pe)
        expected_errors = ph * (1 - pe)
        # Observed errors: fraction of repos where harder=1 but easier=0
        has_harder = cat_presence[harder] == 1
        observed_errors = (cat_presence.loc[has_harder, easier] == 0).mean() * ph if has_harder.sum() > 0 else 0
        if expected_errors > 0:
            h = 1 - (observed_errors / expected_errors)
            h_values.append(h)

overall_H = np.mean(h_values) if h_values else 0
print(f"Mean Loevinger H = {overall_H:.3f}")
print(f"  H >= 0.3: weak scale")
print(f"  H >= 0.4: moderate scale")
print(f"  H >= 0.5: strong scale")

# Heatmap of conditional probabilities
fig = go.Figure(go.Heatmap(
    z=cond_matrix.values.astype(float),
    x=[f"{c} (L{int(CATEGORY_TO_LEVEL[c])})" for c in ordered_by_prev],
    y=[f"{c} (L{int(CATEGORY_TO_LEVEL[c])})" for c in ordered_by_prev],
    colorscale='Viridis', zmin=0, zmax=1,
    text=np.round(cond_matrix.values.astype(float), 2),
    texttemplate="%{text:.2f}", textfont={"size": 9},
))
fig.update_layout(
    title="Conditional Probability Matrix: P(row category | column category present)",
    height=500, width=650,
    xaxis_title="Given this category is present...",
    yaxis_title="...probability of this category",
)
save_fig(fig, "rq1b_conditional_probability")
fig.show()

In [ ]:
# --- Analysis 1b.8: Leave-One-Category-Out Guttman Stability ---
# Remove each category and recompute Guttman CR/CS to check whether
# any single category drives the scale result.

print("=== Leave-One-Category-Out Guttman Stability ===\n")
print(f"{'Removed':<20s} {'CR':>6s} {'CS':>6s} {'CR pass':>8s} {'CS pass':>8s}")
print("-" * 55)

loo_results = []
for drop_cat in CATEGORY_NAMES:
    remaining = [c for c in CATEGORY_NAMES if c != drop_cat]
    # Recompute level assignments with remaining categories
    loo_l2 = [c for c in remaining if CATEGORY_TO_LEVEL[c] == MaturityLevel.L2]
    loo_l3 = [c for c in remaining if CATEGORY_TO_LEVEL[c] == MaturityLevel.L3]
    loo_l4 = [c for c in remaining if CATEGORY_TO_LEVEL[c] == MaturityLevel.L4]
    
    # Skip if a level has no categories left
    if not loo_l2 or not loo_l3 or not loo_l4:
        print(f"{drop_cat:<20s} {'N/A':>6s} {'N/A':>6s} (level emptied)")
        loo_results.append((drop_cat, None, None))
        continue
    
    has_l2 = cat_presence[loo_l2].any(axis=1).astype(int).values
    has_l3 = cat_presence[loo_l3].any(axis=1).astype(int).values
    has_l4 = cat_presence[loo_l4].any(axis=1).astype(int).values
    
    cr, cs = compute_guttman_cr_cs(has_l2, has_l3, has_l4)
    cr_pass = "PASS" if cr >= 0.90 else "FAIL"
    cs_pass = "PASS" if cs >= 0.60 else "FAIL"
    print(f"{drop_cat:<20s} {cr:6.3f} {cs:6.3f} {cr_pass:>8s} {cs_pass:>8s}")
    loo_results.append((drop_cat, cr, cs))

print(f"\n{'(baseline)':<20s} {obs_cr:6.3f} {obs_cs:6.3f}")
print(f"\nStability: scale is robust if all LOO variants pass CR>=0.90 and CS>=0.60.")

# Bar chart
valid_results = [(cat, cr, cs) for cat, cr, cs in loo_results if cr is not None]
fig = make_subplots(rows=1, cols=2, subplot_titles=["CR (leave-one-out)", "CS (leave-one-out)"])
cats_plot = [r[0] for r in valid_results]
crs_plot = [r[1] for r in valid_results]
css_plot = [r[2] for r in valid_results]

fig.add_trace(go.Bar(x=cats_plot, y=crs_plot, marker_color='#3b82f6', name='CR'), row=1, col=1)
fig.add_hline(y=0.90, line_dash="dash", line_color="#ef4444", row=1, col=1,
              annotation_text="CR threshold (0.90)")
fig.add_hline(y=obs_cr, line_dash="dot", line_color="#22c55e", row=1, col=1,
              annotation_text=f"Baseline ({obs_cr:.3f})")

fig.add_trace(go.Bar(x=cats_plot, y=css_plot, marker_color='#f97316', name='CS'), row=1, col=2)
fig.add_hline(y=0.60, line_dash="dash", line_color="#ef4444", row=1, col=2,
              annotation_text="CS threshold (0.60)")
fig.add_hline(y=obs_cs, line_dash="dot", line_color="#22c55e", row=1, col=2,
              annotation_text=f"Baseline ({obs_cs:.3f})")

fig.update_layout(
    title="Leave-One-Category-Out Guttman Stability",
    height=400, width=900, showlegend=False,
)
fig.update_xaxes(tickangle=-45)
save_fig(fig, "rq1b_loo_stability")
fig.show()

---

## Summary & Key Findings

**Sampling frame**: the full 441-repo private frame (27 orgs) — 210 repos scored on strict+W+ artifacts plus 231 whitelist-excluded repos included as L1. All results are validated on the AI-tools-only subset (217 repos with known tool associations) as a robustness check.

In [ ]:
# --- Consolidated Summary Statistics ---
print("=" * 70)
print("RQ1 SUMMARY: Is AI Artifact Adoption Structured or Ad Hoc?")
print("=" * 70)

print(f"\nDataset: {len(repo_scores_df)} repositories, {repo_scores_df['org_name'].nunique()} organizations")
print(f"Robustness subset: {len(ai_scores_df)} repos with known AI tool associations")
print(f"Framework: AIME — 9 categories, 4 maturity levels")
print(f"Sampling frame: full 441-repo filtered population; repos with no strict AI artifacts included as L1")

print(f"\n--- RQ1c: Population Distribution ---")
print(f"Among repos with AI artifacts, maturity is spread across L2-L4:")
for lvl in [2, 3, 4]:
    c = level_counts.get(lvl, 0)
    print(f"  L{lvl} {MATURITY_LABELS[MaturityLevel(lvl)]:25s}: {c:4d} ({c/total*100:.1f}%)")

print(f"\n--- RQ1a: Cumulative Broadening ---")
print(f"  Level-breadth correlation: rho={rho:.3f}, p={p:.2e}")
print(f"  AI-tools subset:          rho={rho_ai:.3f}, p={p_ai:.2e}")
print(f"  Unique category combinations: {len(profile_counts)} (high diversity)")
sil_note = "weak → gradient, not archetypes" if best_sil < 0.30 else "strong → discrete profile clusters (check level alignment)"
print(f"  Clustering silhouette: {best_sil:.3f} ({sil_note})")
print(f"  Conclusion: Adoption follows structured cumulative broadening — repos")
print(f"  build outward from an L2 foundation, not random category sampling.")

print(f"\n--- RQ1b: Cumulative Maturity ---")
print(f"  Guttman CR: {CR:.3f} ({'PASS' if CR >= 0.90 else 'FAIL'} >= 0.90)")
print(f"  Guttman CS: {CS:.3f} ({'PASS' if CS >= 0.60 else 'FAIL'} >= 0.60)")
print(f"  AI-tools subset: CR={CR_ai:.3f}, CS={CS_ai:.3f}")
l3_repos = df[df['level'] == 3]
l4_repos = df[df['level'] == 4]
if len(l3_repos) > 0:
    print(f"  L3 repos with L2 foundation: {l3_repos['has_l2'].mean()*100:.1f}%")
if len(l4_repos) > 0:
    print(f"  L4 repos with L2+L3 foundation: {((l4_repos['has_l2']) & (l4_repos['has_l3'])).mean()*100:.1f}%")

print(f"\n{'=' * 70}")